Загрузка датасета без формирования выборки

In [ ]:
import kagglehub
from pathlib import Path
# Скачивание датасета через kagglehub
# ВАЖНО: если это ваш первый запуск, Colab попросит авторизоваться через Kaggle API token
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
base_dir = Path(path) / "chest_xray"

print(f"Датасет сохранён в: {base_dir}")
print("Структура папок:")
for p in base_dir.iterdir():
    if p.is_dir():
        print(f"  {p.name}/")

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Датасет сохранён в: /kaggle/input/chest-xray-pneumonia/chest_xray
Структура папок:
  chest_xray/
  __MACOSX/
  val/
  test/
  train/


# Grad-Cam

In [ ]:
# import shutil
# shutil.rmtree("outputs")

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision import models, transforms
from safetensors.torch import load_file

# 1. Загружаем CSV тимлида
df = pd.read_csv("test_conformal_results.csv")
print(f"Загружено {len(df)} записей из CSV.")

# 2. Строим словарь: имя_файла.jpeg -> полный_путь
# Сканируем всю папку датасета, чтобы найти файлы гарантированно
image_paths = {}
for img_path in Path(base_dir).rglob("*.jpeg"):
    image_paths[img_path.name] = str(img_path)

print(f"Найдено {len(image_paths)} уникальных изображений в датасете.")

# Проверяем, все ли файлы из CSV существуют
missing = [img_id for img_id in df['image_id'] if img_id not in image_paths]
if missing:
    print(f"⚠ ВНИМАНИЕ: {len(missing)} файлов из CSV не найдены в датасете!")
else:
    print("✅ Все файлы из CSV успешно сопоставлены с путями.")

Загружено 879 записей из CSV.
Найдено 11712 уникальных изображений в датасете.
✅ Все файлы из CSV успешно сопоставлены с путями.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

# Инициализация архитектуры
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 3)

# Загрузка весов
state_dict = load_file("best_model.safetensors")
model.load_state_dict(state_dict)
model = model.to(device)
model.eval() # КРИТИЧЕСКИ ВАЖНО: режим инференса

print("✅ Модель загружена и переведена в режим eval().")

# Отпечаток для проверки воспроизводимости
print("Отпечаток весов (первые 3 значения fc.bias):", list(model.fc.parameters())[1].data[:3].cpu().numpy())

Используем устройство: cuda
✅ Модель загружена и переведена в режим eval().
Отпечаток весов (первые 3 значения fc.bias): [ 0.03466464 -0.03043265  0.00167839]


In [ ]:
# Укажите здесь имя файла, который хотите проверить.
# Можете взять первый из CSV: df.iloc[0]['image_id'] или ваш 'person147_bacteria_706.jpeg'
target_image = "person147_bacteria_711.jpeg" # Замените на 'person147_bacteria_706.jpeg', если он есть в вашем полном CSV

if target_image not in image_paths:
    print(f"❌ Ошибка: файл {target_image} не найден в датасете!")
else:
    print(f"\n🔍 Проверяем изображение: {target_image}")

    # 1. Берем эталонные данные из CSV
    csv_row = df[df['image_id'] == target_image].iloc[0]
    csv_probs = np.array([
        csv_row['prob_normal'],
        csv_row['prob_bacterial'],
        csv_row['prob_viral']
    ])
    print(f"Вероятности из CSV:   [{csv_probs[0]:.6f}, {csv_probs[1]:.6f}, {csv_probs[2]:.6f}]")

    # 2. Прогоняем через модель
    img_path = image_paths[target_image]
    img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # Тот же препроцессинг, что при обучении
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = clahe.apply(img_gray)
    img_rgb = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    input_tensor = transform(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        model_probs = F.softmax(logits, dim=1).cpu().numpy()[0]

    print(f"Вероятности от модели: [{model_probs[0]:.6f}, {model_probs[1]:.6f}, {model_probs[2]:.6f}]")

    # 3. Сравниваем (с учетом погрешности float32)
    max_diff = np.max(np.abs(csv_probs - model_probs))
    print(f"Максимальная разница: {max_diff:.2e}")

    if max_diff < 1e-5:
        print("✅ УСПЕХ: Предсказания модели ИДЕНТИЧНЫ данным в CSV (с точностью до float32).")
    else:
        print("⚠ ВНИМАНИЕ: Есть расхождения! Проверьте препроцессинг или порядок классов.")


🔍 Проверяем изображение: person147_bacteria_711.jpeg
Вероятности из CSV:   [0.130000, 0.570374, 0.299626]
Вероятности от модели: [0.129999, 0.570374, 0.299627]
Максимальная разница: 1.04e-06
✅ УСПЕХ: Предсказания модели ИДЕНТИЧНЫ данным в CSV (с точностью до float32).


Вот тут считается энтропия вместо площади сейчас, прошарь

In [ ]:
CLASS_NAMES = {0: "normal", 1: "bacterial", 2: "viral"}

# Инициализируем Grad-CAM один раз
target_layers = [model.layer4[-1]]
cam_extractor = GradCAM(model=model, target_layers=target_layers)

def compute_entropy(cam_7x7):
    cam_min = cam_7x7.min()
    cam_max = cam_7x7.max()
    if cam_max - cam_min < 1e-6:
        return 0.0, True

    normalized = (cam_7x7 - cam_min) / (cam_max - cam_min)
    cam_sum = normalized.sum()
    if cam_sum < 1e-6:
        return 0.0, True

    p = normalized / cam_sum
    p_safe = np.where(p > 0, p, 1e-10)
    entropy = -np.sum(p_safe * np.log2(p_safe))

    max_entropy = np.log2(49) # Для разрешения 7x7
    return float(entropy / max_entropy), False

def process_single_image(image_id, true_label, set_size, in_set_flags):
    # 1. Загружаем и предобрабатываем изображение
    img_path = image_paths[image_id]
    img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img_clahe = clahe.apply(img_gray)
    img_rgb = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB) # Оригинальный размер (напр. 1555x1944)

    input_tensor = transform(img_rgb).unsqueeze(0).to(device) # Сжатый до 224x224 для модели

    # 2. Определяем, для каких классов строить Grad-CAM
    classes_to_process = [true_label]
    if set_size > 1:
        for class_idx in range(3):
            if in_set_flags[class_idx] and class_idx != true_label:
                classes_to_process.append(class_idx)

    results = []
    stem = Path(image_id).stem

    for target_class in classes_to_process:
        # Строим Grad-CAM (возвращает карту 7x7 для ResNet18)
        targets = [ClassifierOutputTarget(target_class)]
        grayscale_cam = cam_extractor(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0] # shape (7, 7)

        # Считаем энтропию на нативном разрешении 7x7
        entropy, is_degenerate = compute_entropy(grayscale_cam)

        # Сохраняем numpy-массив (нативное разрешение 7x7)
        os.makedirs("outputs/gradcam/arrays", exist_ok=True)
        array_filename = f"{stem}_class_{CLASS_NAMES[target_class]}.npy"
        np.save(f"outputs/gradcam/arrays/{array_filename}", grayscale_cam)

        # === ИСПРАВЛЕНИЕ ЗДЕСЬ ===
        # Получаем оригинальные размеры изображения
        orig_h, orig_w = img_rgb.shape[:2]

        # Растягиваем тепловую карту 7x7 до ОРИГИНАЛЬНОГО размера изображения
        # Используем INTER_CUBIC для более гладкого растягивания
        cam_resized = cv2.resize(grayscale_cam, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)

        # Нормализуем растянутую карту в [0, 1]
        cam_min, cam_max = cam_resized.min(), cam_resized.max()
        if cam_max - cam_min > 1e-6:
            cam_normalized = (cam_resized - cam_min) / (cam_max - cam_min + 1e-6)
        else:
            cam_normalized = np.zeros_like(cam_resized)

        # Готовим оригинальное изображение (должно быть float32 в диапазоне [0, 1])
        rgb_img_normalized = img_rgb.astype(np.float32) / 255.0

        # Накладываем карту. Теперь размеры совпадают!
        visualization = show_cam_on_image(rgb_img_normalized, cam_normalized, use_rgb=True)

        # Сохраняем визуализацию
        os.makedirs("outputs/gradcam/visualizations", exist_ok=True)
        viz_filename = f"{stem}_class_{CLASS_NAMES[target_class]}.png"
        viz_path = f"outputs/gradcam/visualizations/{viz_filename}"
        plt.imsave(viz_path, visualization)
        # =========================

        results.append({
            'image_id': image_id,
            'true_label': true_label,
            'target_class': target_class,
            'target_class_name': CLASS_NAMES[target_class],
            'is_true_class': (target_class == true_label),
            'set_size': set_size,
            'entropy': entropy,
            'is_degenerate': is_degenerate,
            'visualization_path': viz_path,
            'array_path': f"outputs/gradcam/arrays/{array_filename}"
        })

    return results

In [ ]:
os.makedirs("outputs", exist_ok=True)
all_results = []

print(f"Начинаем обработку {len(df)} изображений из тестовой выборки...")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Grad-CAM Inference"):
    image_id = row['image_id']

    # Пропускаем, если файл вдруг не найден (на случай битых ссылок в CSV)
    if image_id not in image_paths:
        continue

    true_label = int(row['true_label'])
    set_size = int(row['set_size'])
    in_set_flags = [
        bool(row['in_set_normal']),
        bool(row['in_set_bacterial']),
        bool(row['in_set_viral'])
    ]

    try:
        results = process_single_image(image_id, true_label, set_size, in_set_flags)
        all_results.extend(results)
    except Exception as e:
        print(f"\nОшибка на {image_id}: {e}")

# Сохранение итоговой таблицы
df_gradcam = pd.DataFrame(all_results)
df_gradcam.to_csv("outputs/gradcam_metrics.csv", index=False)

print("\n" + "="*60)
print("✅ ОБРАБОТКА ЗАВЕРШЕНА!")
print("="*60)
print(f"Всего сгенерировано карт Grad-CAM: {len(df_gradcam)}")
print(f"  - для истинных классов: {df_gradcam['is_true_class'].sum()}")
print(f"  - для ложных классов:   {(~df_gradcam['is_true_class']).sum()}")
print(f"💾 Метрики сохранены в: outputs/gradcam_metrics.csv")

Начинаем обработку 879 изображений из тестовой выборки...


Grad-CAM Inference: 100%|██████████| 879/879 [14:05<00:00,  1.04it/s]


✅ ОБРАБОТКА ЗАВЕРШЕНА!
Всего сгенерировано карт Grad-CAM: 1271
  - для истинных классов: 879
  - для ложных классов:   392
💾 Метрики сохранены в: outputs/gradcam_metrics.csv


# Сохранение результатов Grad-CAM

In [ ]:
import os
from google.colab import files

# Создаём zip-архив папки outputs
!zip -r outputs.zip outputs/

# Показываем размер архива
size_mb = os.path.getsize("outputs.zip") / (1024 * 1024)
print(f"✅ Архив создан: outputs.zip ({size_mb:.2f} MB)")

# Скачиваем на компьютер
from google.colab import drive
drive.mount('/content/drive')

# Копируем архив на Google Drive
!cp outputs.zip /content/drive/MyDrive/outputs.zip
print("✅ Архив скопирован в /MyDrive/outputs.zip")

updating: outputs/ (stored 0%)
updating: outputs/gradcam/ (stored 0%)
updating: outputs/gradcam/arrays/ (stored 0%)
updating: outputs/gradcam/arrays/person928_virus_1586_class_viral.npy (deflated 41%)
updating: outputs/gradcam/arrays/person1453_virus_2485_class_viral.npy (deflated 33%)
updating: outputs/gradcam/arrays/person740_bacteria_2643_class_viral.npy (deflated 35%)
updating: outputs/gradcam/arrays/NORMAL2-IM-1280-0001_class_normal.npy (deflated 33%)
updating: outputs/gradcam/arrays/person1097_virus_1817_class_viral.npy (deflated 39%)
updating: outputs/gradcam/arrays/NORMAL2-IM-0611-0001_class_normal.npy (deflated 76%)
updating: outputs/gradcam/arrays/person810_virus_1445_class_bacterial.npy (deflated 49%)
updating: outputs/gradcam/arrays/person692_virus_1286_class_bacterial.npy (deflated 34%)
updating: outputs/gradcam/arrays/person1843_bacteria_4710_class_viral.npy (deflated 67%)
updating: outputs/gradcam/arrays/person443_bacteria_1924_class_viral.npy (deflated 48%)
updating: ou